# Prediction Visualiser

For each task in a trained checkpoint, runs greedy inference (leave-one-out)
and shows **input | expected output | prediction** side by side.
Wrong cells in the prediction are highlighted with a red border.

**Setup:** set `CHECKPOINT` below to your downloaded `.pt` file, then Run All.

In [ ]:
%matplotlib inline
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import torch

# ── Config ────────────────────────────────────────────────────────────────────
CHECKPOINT = '../checkpoints/transformer_cC8_compartment_fill_best.pt'
K_CONTEXT  = 3      # context pairs per inference call
# ─────────────────────────────────────────────────────────────────────────────

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from src.arc_tokenizer import ArcTokenizer, VOCAB_SIZE
from src.transformer_model import ArcTransformer
from scripts.evaluate import load_checkpoint, greedy_decode

ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

device = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))
print(f'Device: {device}')

model, saved_args, task_ids = load_checkpoint(CHECKPOINT, device)
tok = ArcTokenizer()
print(f'Tasks: {task_ids}')

In [ ]:
def show_grid(ax, grid, title, wrong_mask=None):
    """Render a grid. If wrong_mask provided, draw a red cell border on wrong cells."""
    g = np.array(grid)
    ax.imshow(g, cmap=CMAP, norm=NORM, interpolation='nearest')
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    if wrong_mask is not None:
        H, W = g.shape
        for r in range(H):
            for c in range(W):
                if wrong_mask[r, c]:
                    rect = patches.Rectangle(
                        (c - 0.5, r - 0.5), 1, 1,
                        linewidth=1.5, edgecolor='red', facecolor='none'
                    )
                    ax.add_patch(rect)


DATA_DIR = ROOT / 'data' / 'training'

for tid in task_ids:
    path = DATA_DIR / f'{tid}.json'
    if not path.exists():
        print(f'!! {tid}: file not found')
        continue

    task   = json.loads(path.read_text())
    train  = task['train']
    n      = len(train)
    n_exact = 0
    accs    = []

    print(f'\n{"="*70}')
    print(f'  {tid}  ({n} train pairs)')
    print(f'{"="*70}')

    for hi in range(n):
        ctx_raw = [p for i, p in enumerate(train) if i != hi]
        tp      = train[hi]
        test_in = np.array(tp['input'],  dtype=np.uint8)
        target  = np.array(tp['output'], dtype=np.uint8)
        H, W    = target.shape

        ctx  = [(np.array(p['input'],  dtype=np.uint8),
                 np.array(p['output'], dtype=np.uint8))
                for p in ctx_raw[:K_CONTEXT]]
        pred = greedy_decode(model, tok, ctx, test_in, H, W, device)

        wrong = pred != target
        ca    = float((pred == target).mean())
        em    = bool(np.array_equal(pred, target))
        accs.append(ca)
        if em:
            n_exact += 1

        label = 'EXACT' if em else f'{wrong.sum()} wrong cells'
        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        show_grid(axes[0], test_in, f'[{hi}] Input')
        show_grid(axes[1], target,  f'[{hi}] Expected')
        show_grid(axes[2], pred,    f'[{hi}] Predicted  acc={ca:.3f}  {label}',
                  wrong_mask=wrong if not em else None)
        fig.suptitle(f'{tid}  pair {hi}', fontsize=9, y=1.01)
        plt.tight_layout()
        plt.show()

    mean_ca = np.mean(accs)
    print(f'  → cell_acc={mean_ca:.3f}  exact={n_exact}/{n}')